In [ ]:
from datetime import datetime, timezone
import json
import time

import pandas as pd
import requests

from project_paths import project_root as get_project_root
from pipeline_config import (
    COUNTRY_CODES,
    EXPECTED_COUNTRY_NAMES,
    GEOCODING_QUERY_OVERRIDES,
    OPEN_METEO_GEOCODING_ENDPOINT as GEOCODING_ENDPOINT,
    normalize_for_match,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 30)

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
cache_path = base_path / 'data' / 'cache'
outputs_tables_path = base_path / 'outputs' / 'tables'

venues_output_path = bronze_path / 'venues_unique.parquet'
geocoding_cache_path = cache_path / 'geocoding_cache.json'
geocodes_output_path = silver_path / 'venue_geocodes.parquet'
geocoding_failures_output_path = outputs_tables_path / 'geocoding_failures.csv'

for path in [silver_path, cache_path, outputs_tables_path]:
    path.mkdir(parents=True, exist_ok=True)

venues_unique = pd.read_parquet(venues_output_path)

{
    'venues_input': 'data/bronze/venues_unique.parquet',
    'geocodes_output': 'data/silver/venue_geocodes.parquet',
    'geocoding_cache_output': 'data/cache/geocoding_cache.json',
    'geocoding_failures_output': 'outputs/tables/geocoding_failures.csv',
    'endpoint': GEOCODING_ENDPOINT,
}


# 03b Geocoding vorbereiten

Fuer Wetterabfragen reicht eine Koordinate pro Stadt/Land-Kombination. Deshalb wird der API-Cache auf `location_key` gefuehrt und danach wieder auf die Venue-Zeilen gejoint. Die Abfrage nutzt die Open-Meteo Geocoding API und speichert die Rohentscheidung im JSON-Cache.

In [ ]:
def load_geocoding_cache(path):
    if not path.exists():
        return {}
    with path.open('r', encoding='utf-8') as file:
        return json.load(file)


def save_geocoding_cache(cache, path):
    with path.open('w', encoding='utf-8') as file:
        json.dump(cache, file, ensure_ascii=False, indent=2, sort_keys=True)
        file.write('\n')


def choose_best_geocoding_result(results, city_name, country_name):
    if not results:
        return None, 'no_results'

    expected_country_code = COUNTRY_CODES.get(country_name)
    expected_country_names = EXPECTED_COUNTRY_NAMES.get(country_name, {country_name})
    normalized_city = normalize_for_match(city_name)
    normalized_expected_countries = {normalize_for_match(country) for country in expected_country_names}

    scored_results = []
    for position, result in enumerate(results):
        result_country_code = result.get('country_code')
        result_country_name = result.get('country')
        result_name = result.get('name')
        score = 0
        if expected_country_code and result_country_code == expected_country_code:
            score += 100
        if normalize_for_match(result_country_name) in normalized_expected_countries:
            score += 50
        if normalize_for_match(result_name) == normalized_city:
            score += 25
        score -= position
        scored_results.append((score, position, result))

    scored_results.sort(key=lambda item: item[0], reverse=True)
    best_score, _, best_result = scored_results[0]
    quality_flag = 'ok' if best_score >= 120 else 'review'
    return best_result, quality_flag


def geocode_location(row, cache, request_delay_seconds=0.15):
    location_key = row['location_key']
    city_name = row['city_name']
    country_name = row['country_name']
    query_name = GEOCODING_QUERY_OVERRIDES.get((city_name, country_name), city_name)
    cached_entry = cache.get(location_key)
    if cached_entry is not None and cached_entry.get('query_name', city_name) == query_name:
        entry = dict(cached_entry)
        entry['cache_hit'] = True
        return entry

    country_code = COUNTRY_CODES.get(country_name)
    params = {
        'name': query_name,
        'count': 10,
        'language': 'en',
        'format': 'json',
    }
    if country_code:
        params['countryCode'] = country_code

    entry = {
        'location_key': location_key,
        'city_name': city_name,
        'country_name': country_name,
        'query_name': query_name,
        'country_code': country_code,
        'source': 'open-meteo-geocoding',
        'requested_at_utc': datetime.now(timezone.utc).isoformat(),
        'cache_hit': False,
    }

    try:
        response = requests.get(GEOCODING_ENDPOINT, params=params, timeout=20)
        entry['request_url'] = response.url
        entry['http_status'] = response.status_code
        response.raise_for_status()
        payload = response.json()
        results = payload.get('results', [])
        result, quality_flag = choose_best_geocoding_result(results, city_name, country_name)
        entry['result_count'] = len(results)
        entry['quality_flag'] = quality_flag

        if result is None:
            entry.update({
                'geocode_status': 'not_found',
                'latitude': None,
                'longitude': None,
                'matched_name': None,
                'matched_country': None,
                'matched_country_code': None,
                'matched_admin1': None,
                'failure_reason': 'Open-Meteo returned no results for this city/country.',
            })
        else:
            entry.update({
                'geocode_status': 'found',
                'latitude': result.get('latitude'),
                'longitude': result.get('longitude'),
                'matched_name': result.get('name'),
                'matched_country': result.get('country'),
                'matched_country_code': result.get('country_code'),
                'matched_admin1': result.get('admin1'),
                'matched_timezone': result.get('timezone'),
                'failure_reason': None if quality_flag == 'ok' else 'Best API result should be reviewed before weather extraction.',
            })
    except Exception as error:
        entry.update({
            'geocode_status': 'error',
            'latitude': None,
            'longitude': None,
            'matched_name': None,
            'matched_country': None,
            'matched_country_code': None,
            'matched_admin1': None,
            'matched_timezone': None,
            'result_count': 0,
            'quality_flag': 'error',
            'failure_reason': str(error),
        })

    cache[location_key] = {key: value for key, value in entry.items() if key != 'cache_hit'}
    time.sleep(request_delay_seconds)
    return entry


geocoding_cache = load_geocoding_cache(geocoding_cache_path)
location_requests = (
    venues_unique[['location_key', 'city_name', 'country_name']]
    .drop_duplicates()
    .sort_values(['country_name', 'city_name'])
    .reset_index(drop=True)
)

{
    'unique_locations': len(location_requests),
    'cached_locations_before_run': len(geocoding_cache),
    'endpoint': GEOCODING_ENDPOINT,
}

## Open-Meteo API abfragen und Cache aktualisieren

Die API wird nur fuer Location-Keys abgefragt, die noch nicht im Cache liegen. Bei wiederholter Ausfuehrung kommen die Ergebnisse direkt aus `data/cache/geocoding_cache.json`.

In [ ]:
geocode_entries = [
    geocode_location(row, geocoding_cache)
    for _, row in location_requests.iterrows()
]
save_geocoding_cache(geocoding_cache, geocoding_cache_path)

location_geocodes = pd.DataFrame(geocode_entries)
location_geocodes['latitude'] = pd.to_numeric(location_geocodes['latitude'], errors='coerce')
location_geocodes['longitude'] = pd.to_numeric(location_geocodes['longitude'], errors='coerce')

{
    'locations_requested': len(location_requests),
    'locations_geocoded': int((location_geocodes['geocode_status'] == 'found').sum()),
    'cache_hits_this_run': int(location_geocodes['cache_hit'].sum()),
    'cache_entries_after_run': len(geocoding_cache),
    'status_counts': location_geocodes['geocode_status'].value_counts(dropna=False).to_dict(),
    'quality_counts': location_geocodes['quality_flag'].value_counts(dropna=False).to_dict(),
}

## Venue-Geocodes speichern

Die city-level Geocodes werden zur Venue-Tabelle zurueckgejoint. So bleibt jede Stadionzeile erhalten, nutzt aber denselben Location-Cache wie andere Stadien in derselben Stadt.

In [ ]:
geocode_columns = [
    'location_key',
    'latitude',
    'longitude',
    'geocode_status',
    'quality_flag',
    'source',
    'cache_hit',
    'matched_name',
    'matched_country',
    'matched_country_code',
    'matched_admin1',
    'matched_timezone',
    'result_count',
    'request_url',
    'failure_reason',
]

venue_geocodes = venues_unique.merge(
    location_geocodes[geocode_columns],
    on='location_key',
    how='left',
)
venue_geocodes.to_parquet(geocodes_output_path, index=False)

{
    'venue_rows': len(venue_geocodes),
    'venues_with_coordinates': int(venue_geocodes[['latitude', 'longitude']].notna().all(axis=1).sum()),
    'unique_geocoded_locations': int(location_geocodes.loc[location_geocodes['geocode_status'].eq('found'), 'location_key'].nunique()),
    'geocodes_output': 'data/silver/venue_geocodes.parquet',
}

## Fehlende und unsichere Geocodes dokumentieren

Die Failure-Tabelle enthaelt echte Fehler sowie Ergebnisse mit `quality_flag = review`. Damit sind fehlende oder unsichere Lookups explizit sichtbar, auch wenn die Pipeline weiterlaufen kann.

In [ ]:
failure_mask = (
    location_geocodes['geocode_status'].ne('found')
    | location_geocodes['latitude'].isna()
    | location_geocodes['longitude'].isna()
    | location_geocodes['quality_flag'].ne('ok')
)

failure_columns = [
    'location_key',
    'city_name',
    'country_name',
    'country_code',
    'geocode_status',
    'quality_flag',
    'latitude',
    'longitude',
    'matched_name',
    'matched_country',
    'matched_country_code',
    'matched_admin1',
    'result_count',
    'failure_reason',
    'request_url',
]
geocoding_failures = location_geocodes.loc[failure_mask, failure_columns].copy()
geocoding_failures.to_csv(geocoding_failures_output_path, index=False)

{
    'geocoding_failures_or_reviews': len(geocoding_failures),
    'failure_status_counts': geocoding_failures['geocode_status'].value_counts(dropna=False).to_dict() if not geocoding_failures.empty else {},
    'failure_quality_counts': geocoding_failures['quality_flag'].value_counts(dropna=False).to_dict() if not geocoding_failures.empty else {},
    'geocoding_failures_output': 'outputs/tables/geocoding_failures.csv',
}

## Qualitaetschecks BD-10

Die Checks bilden die BD-10-Akzeptanzkriterien ab: Die meisten Venues haben Koordinaten, der Cache deckt alle Location-Keys ab und fehlende oder unsichere Geocodes werden als CSV dokumentiert.

In [ ]:
coordinate_coverage = venue_geocodes[['latitude', 'longitude']].notna().all(axis=1).mean()
missing_cache_keys = sorted(set(location_requests['location_key']) - set(geocoding_cache.keys()))
failed_locations = location_geocodes[location_geocodes['geocode_status'].ne('found')]

quality_checks_bd10 = {
    'geocodes_output_exists': geocodes_output_path.exists(),
    'geocoding_cache_exists': geocoding_cache_path.exists(),
    'geocoding_failures_output_exists': geocoding_failures_output_path.exists(),
    'venue_rows': len(venue_geocodes),
    'location_rows': len(location_requests),
    'coordinate_coverage': round(float(coordinate_coverage), 4),
    'missing_cache_keys': missing_cache_keys,
    'failed_locations': failed_locations[['location_key', 'failure_reason']].to_dict('records'),
}

assert quality_checks_bd10['geocodes_output_exists']
assert quality_checks_bd10['geocoding_cache_exists']
assert quality_checks_bd10['geocoding_failures_output_exists']
assert quality_checks_bd10['venue_rows'] == len(venues_unique)
assert coordinate_coverage >= 0.8
assert not missing_cache_keys
assert len(failed_locations) == 0

quality_checks_bd10

In [ ]:
venue_geocodes.head(20)

## Ergebnis BD-10

BD-10 ist erfuellt, wenn die Assertions erfolgreich sind und `data/silver/venue_geocodes.parquet`, `data/cache/geocoding_cache.json` sowie `outputs/tables/geocoding_failures.csv` existieren. Wiederholte Ausfuehrungen nutzen den Cache auf `location_key`-Ebene, damit gleiche Stadt-/Land-Kombinationen nicht mehrfach bei der API angefragt werden.